# Tutorial 11A: Introduction to SFA — SOLUTIONS

Complete solutions to all exercises from the main tutorial notebook.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
from pathlib import Path

import statsmodels.api as sm
from scipy import stats

from panelbox.frontier import StochasticFrontier
from panelbox.frontier.tests import inefficiency_presence_test

np.random.seed(42)
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "outputs" / "figures" / "01_intro"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

In [ ]:
hospital_data = pd.read_csv(DATA_DIR / "hospital_data.csv")
farm_data = pd.read_csv(DATA_DIR / "farm_data.csv")
print(f"Hospital data: {hospital_data.shape}")
print(f"Farm data: {farm_data.shape}")

---
## Exercise 1: Agricultural Efficiency Analysis

**Task**: Load the `farm_data.csv` dataset and:
1. Estimate a Cobb-Douglas production frontier
2. Test for presence of inefficiency
3. Calculate BC efficiency scores
4. Identify the 10 most and least efficient farms
5. Analyze efficiency by farm size category

In [ ]:
print("Farm Data Summary")
print("=" * 50)
display(farm_data.head(10))
display(farm_data.describe())
print(f"\nSize categories:\n{farm_data['size_category'].value_counts()}")

In [ ]:
exog_vars_farm = ["log_land", "log_labor", "log_capital", "irrigation"]
X_farm = sm.add_constant(farm_data[exog_vars_farm])
y_farm = farm_data["log_output"]
ols_farm = sm.OLS(y_farm, X_farm).fit()
print(ols_farm.summary())

# Check skewness
skew = stats.skew(ols_farm.resid)
print(f"\nResidual Skewness: {skew:.4f}")

In [ ]:
sfa_farm = StochasticFrontier(
    data=farm_data,
    depvar="log_output",
    exog=["log_land", "log_labor", "log_capital", "irrigation"],
    frontier="production",
    dist="half_normal",
)
result_farm = sfa_farm.fit(method="mle", optimizer="L-BFGS-B")
print(result_farm.summary())

In [ ]:
test_farm = inefficiency_presence_test(
    loglik_sfa=result_farm.loglik,
    loglik_ols=ols_farm.llf,
    residuals_ols=ols_farm.resid.values,
    frontier_type="production",
    distribution="half_normal",
)
print("Test for Inefficiency Presence")
print("=" * 50)
print(f"LR Statistic: {test_farm['lr_statistic']:.4f}")
print(f"P-value: {test_farm['pvalue']:.4f}")
print(f"Conclusion: {test_farm['conclusion']}")

In [ ]:
eff_farm = result_farm.efficiency(estimator="bc", ci_level=0.95)
farm_data["te_bc"] = eff_farm["efficiency"].values

print("Efficiency Summary")
print("=" * 50)
display(farm_data["te_bc"].describe())

# Top 10
print("\nTop 10 Most Efficient Farms:")
top10 = farm_data.nlargest(10, "te_bc")[["farm_id", "te_bc", "size_category", "irrigation"]]
display(top10)

# Bottom 10
print("\nBottom 10 Least Efficient Farms:")
bottom10 = farm_data.nsmallest(10, "te_bc")[["farm_id", "te_bc", "size_category", "irrigation"]]
display(bottom10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot by size
farm_data.boxplot(column="te_bc", by="size_category", ax=axes[0])
axes[0].set_title("Efficiency by Farm Size")
axes[0].set_xlabel("Size Category")
axes[0].set_ylabel("Technical Efficiency (BC)")
axes[0].get_figure().suptitle("")

# Distribution by size
for cat in ["small", "medium", "large"]:
    subset = farm_data[farm_data["size_category"] == cat]["te_bc"]
    axes[1].hist(subset, bins=15, alpha=0.5, label=f"{cat} (mean={subset.mean():.3f})")
axes[1].set_xlabel("Technical Efficiency (BC)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Efficiency Distribution by Farm Size")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "exercise1_farm_efficiency.png", dpi=300, bbox_inches="tight")
plt.show()

# Kruskal-Wallis test
from scipy.stats import kruskal

groups = [
    farm_data[farm_data["size_category"] == cat]["te_bc"] for cat in ["small", "medium", "large"]
]
stat, pval = kruskal(*groups)
print("\nKruskal-Wallis Test (efficiency across size categories):")
print(f"  Statistic: {stat:.4f}")
print(f"  P-value: {pval:.4f}")

### Interpretation

- The SFA model reveals significant inefficiency in agricultural production
- Farm size may/may not significantly affect efficiency levels
- Irrigation access appears to shift the frontier upward
- The efficiency distribution shows the typical pattern: most farms moderately efficient, few extremely inefficient

---
## Exercise 2: Sensitivity Analysis

**Task**: Using the hospital data, compare SFA results with and without control variables, and check if efficiency rankings are robust.

In [ ]:
# Model without controls
sfa_no_controls = StochasticFrontier(
    data=hospital_data,
    depvar="log_cases",
    exog=["log_doctors", "log_nurses", "log_beds"],
    frontier="production",
    dist="half_normal",
)
result_no_controls = sfa_no_controls.fit(method="mle", optimizer="L-BFGS-B")

# Model with controls
sfa_with_controls = StochasticFrontier(
    data=hospital_data,
    depvar="log_cases",
    exog=["log_doctors", "log_nurses", "log_beds", "teaching", "urban"],
    frontier="production",
    dist="half_normal",
)
result_with_controls = sfa_with_controls.fit(method="mle", optimizer="L-BFGS-B")

# Compare
eff_no = result_no_controls.efficiency(estimator="bc")["efficiency"].values
eff_with = result_with_controls.efficiency(estimator="bc")["efficiency"].values

print("Sensitivity Analysis: Effect of Control Variables")
print("=" * 60)

comparison = pd.DataFrame(
    {
        "Statistic": [
            "Mean TE",
            "Std TE",
            "Min TE",
            "Max TE",
            "sigma_v",
            "sigma_u",
            "gamma",
            "Log-Lik",
            "AIC",
        ],
        "Without Controls": [
            f"{eff_no.mean():.4f}",
            f"{eff_no.std():.4f}",
            f"{eff_no.min():.4f}",
            f"{eff_no.max():.4f}",
            f"{result_no_controls.sigma_v:.4f}",
            f"{result_no_controls.sigma_u:.4f}",
            f"{result_no_controls.gamma:.4f}",
            f"{result_no_controls.loglik:.2f}",
            f"{result_no_controls.aic:.2f}",
        ],
        "With Controls": [
            f"{eff_with.mean():.4f}",
            f"{eff_with.std():.4f}",
            f"{eff_with.min():.4f}",
            f"{eff_with.max():.4f}",
            f"{result_with_controls.sigma_v:.4f}",
            f"{result_with_controls.sigma_u:.4f}",
            f"{result_with_controls.gamma:.4f}",
            f"{result_with_controls.loglik:.2f}",
            f"{result_with_controls.aic:.2f}",
        ],
    }
)
display(comparison)

# Rank correlation
from scipy.stats import spearmanr

rho, pval = spearmanr(eff_no, eff_with)
print(f"\nSpearman Rank Correlation: {rho:.4f} (p={pval:.4f})")

# Scatter plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(eff_no, eff_with, alpha=0.6)
ax.plot([0, 1], [0, 1], "r--", label="45-degree line")
ax.set_xlabel("TE without Controls")
ax.set_ylabel("TE with Controls")
ax.set_title("Sensitivity of Efficiency to Control Variables")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "exercise2_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation

- High rank correlation means efficiency rankings are robust to the inclusion of control variables
- Omitting relevant controls may inflate the estimated inefficiency (confounding inefficiency with systematic differences)
- The model with controls is preferred if AIC/BIC are lower

---
## Exercise 3: Returns to Scale Analysis

**Task**: Test whether hospitals exhibit constant returns to scale and discuss economic implications.

In [ ]:
# Use the result with controls
rts_test = result_with_controls.returns_to_scale_test(
    input_vars=["log_doctors", "log_nurses", "log_beds"]
)

print("Returns to Scale Analysis")
print("=" * 60)
print(f"Estimated RTS: {rts_test['rts']:.4f}")
print(f"Standard Error: {rts_test['rts_se']:.4f}")
print(f"95% CI: ({rts_test['ci'][0]:.4f}, {rts_test['ci'][1]:.4f})")
print("\nWald Test (H0: RTS = 1):")
print(f"  Test Statistic: {rts_test['test_statistic']:.4f}")
print(f"  P-value: {rts_test['pvalue']:.4f}")
print(f"  Conclusion: {rts_test['conclusion']}")

# Individual elasticities
input_vars = ["log_doctors", "log_nurses", "log_beds"]
print("\nIndividual Elasticities:")
for var in input_vars:
    coef = result_with_controls.params[var]
    se = result_with_controls.se[var]
    print(f"  {var}: {coef:.4f} (SE: {se:.4f})")

print(f"\nSum of elasticities: {sum(result_with_controls.params[v] for v in input_vars):.4f}")

# Economic interpretation
print("\nEconomic Implications:")
if rts_test["conclusion"] == "DRS":
    print("  - Hospitals exhibit decreasing returns to scale")
    print("  - Doubling all inputs yields less than double the output")
    print("  - Smaller hospitals may be more productive per unit of input")
    print("  - Policy: Consider optimal hospital size rather than unlimited expansion")
elif rts_test["conclusion"] == "CRS":
    print("  - Hospitals exhibit constant returns to scale")
    print("  - Proportional increases in inputs yield proportional output increases")
    print("  - Hospital size does not inherently affect productivity")
elif rts_test["conclusion"] == "IRS":
    print("  - Hospitals exhibit increasing returns to scale")
    print("  - Larger hospitals are more productive per unit of input")
    print("  - Policy: Consolidation may improve overall efficiency")

### Interpretation

The returns to scale estimate tells us about the optimal scale of hospital operations:
- **DRS (RTS < 1)**: Beyond a certain size, adding more inputs has diminishing returns. This suggests an optimal hospital size exists.
- **CRS (RTS = 1)**: Scale doesn't matter — doubling inputs doubles output.
- **IRS (RTS > 1)**: Larger hospitals benefit from economies of scale.

This has direct policy implications for hospital planning and resource allocation.

---
## Discussion Questions

1. Why might the half-normal distribution be too restrictive for modeling hospital inefficiency?
2. How would results change if we used a cost frontier instead of a production frontier?
3. What additional variables might help explain efficiency differences between hospitals?
4. How should policymakers use efficiency scores — should they penalize inefficient hospitals?